#### 03 - Silver Layer: Chicago Food Inspections
**Medallion Architecture — Silver Zone**

Steps:
1. Install and import DQX
2. Read from Bronze
3. DQX Profile — document data quality
4. DQX Expectations — define and apply rules, split good/bad rows
5. Standardize schema
6. Parse violations
7. Write to Silver
8. Verify

- Source: `bronze.bronze_chicago_inspections`
- Target: `silver.silver_chicago_inspections` + `silver.silver_chicago_violations`

##### Step 1: Install DQX

In [0]:
%pip install databricks-labs-dqx -q

In [0]:
dbutils.library.restartPython()

##### Step 2: Configuration & Setup

In [0]:
from databricks.labs.dqx.engine import DQEngine
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, upper, initcap, to_date,
    when, lit, regexp_extract, split,
    explode, monotonically_increasing_id,
    current_timestamp, concat_ws
)

BRONZE_TABLE = "bronze.bronze_chicago_inspections"
SILVER_DB    = "silver"
SILVER_TABLE = "silver_chicago_inspections"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print(f"Source : {BRONZE_TABLE}")
print(f"Target : {SILVER_DB}.{SILVER_TABLE}")

##### Step 3: Read from Bronze

In [0]:
df_bronze = spark.table(BRONZE_TABLE)
print(f"Bronze row count : {df_bronze.count():,}")
print(f"Columns          : {len(df_bronze.columns)}")
display(df_bronze.limit(3))

##### Step 4: DQX Profiling
Profile the bronze data to document quality findings before cleansing.

In [0]:
# DQX Profiling — document data quality before cleansing
df_bronze = spark.table(BRONZE_TABLE)
bronze_count = df_bronze.count()
print(f"Bronze row count: {bronze_count:,}")

profile_cols = ["dba_name", "inspection_date", "inspection_type",
                "zip", "results", "violations", "facility_type", "risk"]

print("\n=== Null Count Profile ===")
for c in profile_cols:
    null_ct = df_bronze.filter(col(c).isNull()).count()
    pct = null_ct / bronze_count * 100
    print(f"  {c:25s} : {null_ct:,} nulls ({pct:.2f}%)")

print("\n=== Unique Value Counts ===")
for c in ["results", "risk", "inspection_type"]:
    uniq = df_bronze.select(c).distinct().count()
    print(f"  {c:25s} : {uniq} unique values")

display(df_bronze.limit(3))

##### Step 5: DQX Expectations — Define Rules
Rules per assignment requirements:
- `dba_name` (Restaurant Name) cannot be null
- `inspection_date` cannot be null
- `inspection_type` cannot be null
- `zip` cannot be null and must match 5-digit format
- `results` (Chicago result) cannot be null

In [0]:
checks = [
    # Rule 1: Restaurant name not null
    {
        "name": "dba_name_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "dba_name"}
        }
    },
    # Rule 2: Inspection date not null
    {
        "name": "inspection_date_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "inspection_date"}
        }
    },
    # Rule 3: Inspection type not null
    {
        "name": "inspection_type_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "inspection_type"}
        }
    },
    # Rule 4: Zip not null
    {
        "name": "zip_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "zip"}
        }
    },
    # Rule 5: Results not null (Chicago-specific)
    {
        "name": "results_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "results"}
        }
    },
]

print(f"DQX rules defined: {len(checks)}")
for c in checks:
    print(f"  - {c['name']}")

##### Step 6: Apply DQX Expectations — Split Good/Bad Rows

In [0]:
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
dqe = DQEngine(ws)

df_good, df_bad = dqe.apply_checks_by_metadata_and_split(
    df_bronze, checks
)

good_count   = df_good.count()
bad_count    = df_bad.count()
bronze_count = df_bronze.count()

print("=== DQX Results ===")
print(f"  Bronze rows  : {bronze_count:,}")
print(f"  ✅ Good rows : {good_count:,}")
print(f"  ❌ Bad rows  : {bad_count:,}  ({bad_count/bronze_count*100:.2f}% dropped)")

print("\nSample bad rows:")
display(df_bad.select("dba_name", "inspection_date", "zip", "results", "_errors").limit(10))

In [0]:
# Manual zip format validation (DQX matches_regex not available in this version)
# Filter out non 5-digit zips from good rows
from pyspark.sql.functions import regexp_extract

zip_bad = df_good.filter(
    ~col("zip").cast("string").rlike("^[0-9]{5}$")
).count()

df_good = df_good.filter(
    col("zip").cast("string").rlike("^[0-9]{5}$")
)
print(f"Additional rows dropped for invalid zip format: {zip_bad}")
print(f"Good rows after zip validation: {df_good.count():,}")

##### Step 7: Standardize Schema on Good Rows

In [0]:
df_silver = (
    df_good
    # Standardize text
    .withColumn("dba_name",        trim(upper(col("dba_name"))))
    .withColumn("aka_name",        trim(upper(col("aka_name"))))
    .withColumn("facility_type",   initcap(trim(col("facility_type"))))
    .withColumn("inspection_type", initcap(trim(col("inspection_type"))))
    .withColumn("results",         initcap(trim(col("results"))))
    .withColumn("risk",            trim(col("risk")))
    .withColumn("address",         trim(upper(col("address"))))
    # Convert inspection date to DATE type
    .withColumn("inspection_date", to_date(col("inspection_date"), "MM/dd/yyyy"))
    # Derive violation score from result (per assignment table)
    .withColumn("derived_score",
        when(col("results") == "Pass", 90)
        .when(col("results") == "Pass W/ Conditions", 80)
        .when(col("results") == "Fail", 70)
        .when(col("results") == "No Entry", 0)
        .otherwise(None)
    )
    # Silver metadata
    .withColumn("_silver_timestamp", current_timestamp())
    .withColumn("_source_city",      lit("Chicago"))
    # Drop DQX internal columns
    .drop("_errors", "_warnings")
)

print(f"Silver row count: {df_silver.count():,}")
display(df_silver.select(
    "dba_name", "inspection_date", "inspection_type",
    "results", "derived_score", "zip", "risk"
).limit(5))

##### Step 8: Parse & Explode Violations
Chicago violations are pipe-separated: `'38. DESC - Comments: MEMO | 47. DESC...'`
Parse into code, description, memo. Deduplicate per inspection.

In [0]:
df_with_violations = (
    df_silver
    .filter(col("violations").isNotNull() & (col("violations") != ""))
    .withColumn("_row_id", monotonically_increasing_id())
)

# Explode pipe-separated string into one row per violation
df_exploded = (
    df_with_violations
    .withColumn("violation_raw", explode(split(col("violations"), r"\|")))
    .withColumn("violation_raw", trim(col("violation_raw")))
    .filter(col("violation_raw") != "")
)

# Parse code, description, memo from each violation string
df_violations = (
    df_exploded
    .withColumn("violation_code",
        regexp_extract(col("violation_raw"), r'^(\d+)\.', 1)
    )
    .withColumn("violation_description",
        regexp_extract(col("violation_raw"), r'^\d+\.\s+(.+?)(?:\s+-\s+Comments:.*)?$', 1)
    )
    .withColumn("violation_memo",
        regexp_extract(col("violation_raw"), r'-\s+Comments:\s+(.+)$', 1)
    )
    .withColumn("source_city", lit("Chicago"))
    # Deduplicate violations per inspection
    .dropDuplicates(["inspection_id", "violation_code"])
    .filter(col("violation_code") != "")
    .select(
        "inspection_id", "dba_name", "inspection_date",
        "violation_code", "violation_description",
        "violation_memo", "source_city"
    )
)

print(f"Total violation rows: {df_violations.count():,}")
display(df_violations.limit(5))

##### Step 9: Write Silver Tables

In [0]:
# Main inspections table
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}")
)
print(f"✅ Written: {SILVER_DB}.{SILVER_TABLE}")

# Violations parsed table
(
    df_violations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.silver_chicago_violations")
)
print(f"✅ Written: {SILVER_DB}.silver_chicago_violations")

##### Step 10: Verify

In [0]:
insp_count = spark.table(f"{SILVER_DB}.{SILVER_TABLE}").count()
viol_count = spark.table(f"{SILVER_DB}.silver_chicago_violations").count()

print(f"✅ Silver inspections  : {insp_count:,}")
print(f"✅ Silver violations   : {viol_count:,}")
print(f"   Rows dropped by DQX : {bronze_count - insp_count:,}")

# Null check on key fields (should all be 0)
df_s = spark.table(f"{SILVER_DB}.{SILVER_TABLE}")
for field in ["dba_name", "inspection_date", "inspection_type", "results"]:
    null_count = df_s.filter(col(field).isNull()).count()
    status = "✅" if null_count == 0 else "❌"
    print(f"   {status} Nulls in {field}: {null_count}")

display(df_s.limit(5))